<a href="https://colab.research.google.com/github/momo54/large_scale_data_management/blob/main/Spark_RDD_Cache_Catalyst.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Spark demos

The code of the slides *Cluster computing with Apache Spark* (Large Scale Data Management).
Run the cells in order, on **Google Colab** or in a **local Jupyter** (Python 3, a Java runtime).

- The first cell installs PySpark 3.5.3 if needed; the data are downloaded or generated by the notebook.
- Times differ from the slides: they were measured on a laptop with `local[4]`.
- Local Jupyter only: the Spark UI is on http://localhost:4040 (jobs, stages, storage, SQL). On Colab it is not reachable from your browser; the cells read its numbers through its REST API.

## 0. Setup

In [ ]:
import os, shutil, subprocess, sys

IN_COLAB = "google.colab" in sys.modules
try:
    import pyspark
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyspark==3.5.3"], check=True)
    import pyspark
if shutil.which("java") is None:                  # Colab normally has Java; install it if not
    subprocess.run("apt-get -qq update && apt-get -qq install -y openjdk-17-jdk-headless > /dev/null", shell=True, check=True)
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
print("pyspark", pyspark.__version__, "| Colab:", IN_COLAB)

In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .master("local[*]")                       # no cluster: one JVM, one task thread per core
         .appName("spark-demos")
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.driver.bindAddress", "127.0.0.1")    # local mode: avoids a BindException
         .config("spark.driver.host", "127.0.0.1")           # when the machine's name has a stale IP
         .config("spark.jars.packages", "graphframes:graphframes:0.8.3-spark3.5-s_2.12")   # section 7
         .config("spark.jars.repositories", "https://repos.spark-packages.org")
         .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("ERROR")
print("Spark", spark.version, "| master", sc.master, "| cores", sc.defaultParallelism, "| UI", sc.uiWebUrl)

What a piece of code cost (jobs, stages, tasks, shuffle), read from the Spark UI's REST API:

In [ ]:
import json, time, urllib.request

UI = "http://localhost:" + sc.uiWebUrl.rsplit(":", 1)[1]

def _rest(path):
    with urllib.request.urlopen(f"{UI}/api/v1/applications/{sc.applicationId}/{path}") as r:
        return json.load(r)

def mb(n):
    return f"{n / 1e6:.1f} MB" if n >= 1e5 else f"{n / 1e3:.1f} kB"

class Measure:
    """with Measure() as m: ...   then print(m)"""
    def __enter__(self):
        time.sleep(0.5)
        self._stages = {(s["stageId"], s["attemptId"]) for s in _rest("stages?status=complete")}
        self._jobs = {j["jobId"] for j in _rest("jobs")}
        self._t = time.time()
        return self
    def __exit__(self, *exc):
        self.seconds = time.time() - self._t
        time.sleep(1.0)                            # let the UI catch up
        new = [s for s in _rest("stages?status=complete") if (s["stageId"], s["attemptId"]) not in self._stages]
        jobs = [j for j in _rest("jobs") if j["jobId"] not in self._jobs]
        self.jobs, self.stages = len(jobs), len(new)
        self.skipped = sum(j["numSkippedStages"] for j in jobs)
        self.tasks = sum(s["numTasks"] for s in new)
        self.shuffle_write = sum(s["shuffleWriteBytes"] for s in new)
        return False
    def __str__(self):
        return (f"{self.jobs} job(s), {self.stages} stage(s) run, {self.skipped} skipped, "
                f"{self.tasks} tasks, shuffle write {mb(self.shuffle_write)}, {self.seconds:.2f} s")

def storage():
    """The Storage tab: one line per cached RDD or DataFrame."""
    return [{k: r[k] for k in ("name", "storageLevel", "numCachedPartitions", "numPartitions", "memoryUsed", "diskUsed")}
            for r in _rest("storage/rdd")]

The data: Moby Dick, the PageRank graph of the MapReduce benchmark (same generator, same seed), the join exercise, the movie documents.

In [ ]:
import json, os, random, urllib.request

os.makedirs("data/movies", exist_ok=True)
BOOK = "data/pg2701.txt"                           # Moby Dick, Project Gutenberg #2701
if not os.path.exists(BOOK):
    urllib.request.urlretrieve("https://www.gutenberg.org/cache/epub/2701/pg2701.txt", BOOK)

GRAPH = "data/graph.tsv"                           # page \t link1,link2,...
N_PAGES = 500_000                                  # 100_000 for a faster run
if not os.path.exists(GRAPH):                      # make_graph.py of the MapReduce benchmark
    random.seed(42)
    with open(GRAPH, "w") as g:
        for page in range(1, N_PAGES + 1):
            k = min(200, max(1, int(random.paretovariate(1.5) * 10 / 3)))
            links = {random.randint(1, N_PAGES) if random.random() < 0.67 else 1 + int(random.paretovariate(1.1)) % 1000
                     for _ in range(k)} - {page}
            links = sorted(links) or [page % N_PAGES + 1]
            g.write(f"{page}\t{','.join(map(str, links))}\n")

with open("data/clients.csv", "w") as f:
    f.write('1,Dupont,Paris\n2,Martin,Lyon\n3,Bernard,Marseille\n')
with open("data/commandes.csv", "w") as f:
    f.write('101,1,Ordinateur,999.99\n102,2,Téléphone,699.99\n103,1,Tablette,399.99\n')
MOVIES = [{"title": "A History of Violence", "year": "2005", "genre": "Crime", "summary": "Tom Stall, a humble family man and owner of a popular neighborhood restaurant, lives a quiet but fulfilling existence in the Midwest. One night Tom foils a crime at his place of business and, to his chagrin, is plastered all over the news for his heroics. Following this, mysterious people follow the Stalls' every move, concerning Tom more than anyone else. As this situation is confronted, more lurks out over where all these occurrences have stemmed from compromising his marriage, family relationship and the main characters' former relations in the process.", "country": "USA", "director": {"last_name": "Cronenberg", "first_name": "David", "birth_date": "1943"}, "actors": [{"first_name": "Ed", "last_name": "Harris", "birth_date": "1950", "role": "Carl Fogarty"}, {"first_name": "Vigo", "last_name": "Mortensen", "birth_date": "1958", "role": "Tom Stall"}, {"first_name": "Maria", "last_name": "Bello", "birth_date": "1967", "role": "Eddie Stall"}, {"first_name": "William", "last_name": "Hurt", "birth_date": "1950", "role": "Richie Cusack"}]}, {"title": "Heat", "year": "1995", "genre": "Crime", "summary": "A group of professional bank robbers start to feel the heat from police when they unknowingly leave a clue at their latest heist.", "country": "USA", "director": {"last_name": "Mann", "first_name": "Michael", "birth_date": "1943"}, "actors": [{"first_name": "Al", "last_name": "Pacino", "birth_date": "1940", "role": "Vincent Hanna"}, {"first_name": "Robert", "last_name": "De Niro", "birth_date": "1943", "role": "Neil McCauley"}, {"first_name": "Val", "last_name": "Kilmer", "birth_date": "1959", "role": "Chris Shiherlis"}]}, {"title": "Marie Antoinette", "year": "2006", "genre": "Drama", "summary": "Based on Antonia Fraser's book about the ill-fated Archduchess of Austria and later Queen of France, 'Marie Antoinette' tells the story of the most misunderstood and abused woman in history, from her birth in Imperial Austria to her later life in France.", "country": "USA", "director": {"last_name": "Coppola", "first_name": "Sofia", "birth_date": "1971"}, "actors": [{"first_name": "Kirsten", "last_name": "Dunst", "birth_date": "1982", "role": "Marie Antoinette"}, {"first_name": "Jason", "last_name": "Schwartzman", "birth_date": "1980", "role": "Louis XVI"}]}, {"title": "Spider-Man", "year": "2002", "genre": "Action", "summary": "On a school field trip, Peter Parker (Maguire) is bitten by a genetically modified spider. He wakes up the next morning with incredible powers. After witnessing the death of his uncle (Robertson), Parkers decides to put his new skills to use in order to rid the city of evil, but someone else has other plans. The Green Goblin (Dafoe) sees Spider-Man as a threat and must dispose of him.", "country": "USA", "director": {"last_name": "Raimi", "first_name": "Sam", "birth_date": "1959"}, "actors": [{"first_name": "Tobey", "last_name": "Maguire", "birth_date": "1975", "role": "Spider-Man / Peter Parker"}, {"first_name": "Kirsten", "last_name": "Dunst", "birth_date": "1982", "role": "Mary Jane Watson"}, {"first_name": "Willem", "last_name": "Dafoe", "birth_date": "1955", "role": "Green Goblin / Norman Osborn"}]}, {"title": "The Social network", "year": "2010", "genre": "Drama", "summary": "tagazou", "country": "USA", "director": {"last_name": "Fincher", "first_name": "David", "birth_date": "1962"}, "actors": [{"first_name": "Jesse", "last_name": "Eisenberg", "birth_date": "1983", "role": "Mark Zuckerberg"}, {"first_name": "Rooney", "last_name": "Mara", "birth_date": "1985", "role": "Erica Albright"}, {"first_name": "Andrew", "last_name": "Garfield", "birth_date": "1983", "role": "Eduardo Saverin "}, {"first_name": "Justin", "last_name": "Timberlake", "birth_date": "1981", "role": "Sean Parker"}]}, {"title": "Unforgiven", "year": "1992", "genre": "Western", "summary": "The town of Big Whisky is full of normal people trying to lead quiet lives. Cowboys try to make a living. Sheriff 'Little Bill' tries to build a house and keep a heavy-handed order. The town whores just try to get by.Then a couple of cowboys cut up a whore. Unsatisfied with Bill's justice, the prostitutes put a bounty on the cowboys. The bounty attracts a young gun billing himself as 'The Schofield Kid', and aging killer William Munny. Munny reformed for his young wife, and has been raising crops and two children in peace. But his wife is gone. Farm life is hard. And Munny is no good at it. So he calls his old partner Ned, saddles his ornery nag, and rides off to kill one more time, blurring the lines between heroism and villainy, man and myth.", "country": "USA", "director": {"last_name": "Eastwood", "first_name": "Clint", "birth_date": "1930"}, "actors": [{"first_name": "Clint", "last_name": "Eastwood", "birth_date": "1930", "role": "William Munny"}, {"first_name": "Gene", "last_name": "Hackman", "birth_date": "1930", "role": "Little Bill Dagget"}, {"first_name": "Morgan", "last_name": "Freeman", "birth_date": "1937", "role": "Ned Logan"}]}]
for m in MOVIES:
    with open(f"data/movies/{m['title'].replace(' ', '_')}.json", "w") as f:
        json.dump(m, f, ensure_ascii=False, indent=1)
print(os.path.getsize(BOOK), "bytes of Moby Dick |", sum(1 for _ in open(GRAPH)), "pages |", len(MOVIES), "movies")

## 1. RDDs: partitions, transformations, actions

In [ ]:
rdd = sc.parallelize(range(16), 4)
print(rdd.getNumPartitions())
for i, part in enumerate(rdd.glom().collect()):
    print(f"Partition {i}: {part}")

A transformation returns at once, an action runs a job:

In [ ]:
with Measure() as m:
    evens = rdd.filter(lambda x: x % 2 == 0)       # transformation
print(evens, "|", m)
with Measure() as m:
    print(evens.collect())                         # action
print(m)

In [ ]:
from operator import add

squares = rdd.map(lambda x: x * x)
print("count  ", squares.count())
print("take(3)", squares.take(3))
print("reduce ", squares.reduce(add))

Word count: `flatMap` + `map` is the map phase, `reduceByKey` the shuffle and the reduce (with a combiner).

In [ ]:
lines = sc.textFile(BOOK, 4)
counts = (lines.flatMap(lambda line: line.lower().split())
               .map(lambda w: (w, 1))
               .reduceByKey(add))
with Measure() as m:
    print(counts.takeOrdered(5, key=lambda kv: -kv[1]))
print(m)

## 2. Looking inside an RDD: `glom()`, `getNumPartitions()`, `toDebugString()`, `partitioner`, `mapPartitions()`

In [ ]:
small = sc.parallelize(["the cat sat", "the dog", "a cat", "the end"], 2)
print(small.getNumPartitions(), small.glom().collect())
words = small.flatMap(lambda l: l.split())
print(words.getNumPartitions(), words.glom().collect())

In [ ]:
wc = words.map(lambda w: (w, 1)).reduceByKey(add, 3)
print(wc.getNumPartitions(), wc.glom().collect())
print("partitioner:", wc.partitioner.numPartitions, "partitions,", wc.partitioner.partitionFunc.__name__)

The lineage: read it bottom-up; `+-` is a shuffle, `(2)` and `(3)` the numbers of partitions.

In [ ]:
print(wc.toDebugString().decode())

In [ ]:
print(words.mapPartitions(lambda it: [sum(1 for _ in it)]).collect())
print(words.mapPartitionsWithIndex(lambda i, it: [(i, list(it))]).collect())

`partitionBy(3)`: key k goes to partition hash(k) mod 3. `mapValues` and `filter` keep the partitioner, `map` forgets it (the key may change).

In [ ]:
pairs = sc.parallelize([(0, "a"), (1, "b"), (2, "c"), (3, "d"), (4, "e"), (5, "f")], 2)
print(pairs.glom().collect())
by3 = pairs.partitionBy(3)
print(by3.glom().collect())

def describe(rdd):
    p = rdd.partitioner
    return "None" if p is None else f"hash of the key, {p.numPartitions} partitions"

for name, r in [("pairs", pairs), ("by3", by3), ("by3.mapValues(str.upper)", by3.mapValues(str.upper)),
                ("by3.filter(lambda kv: kv[0] > 0)", by3.filter(lambda kv: kv[0] > 0)),
                ("by3.map(lambda kv: kv)", by3.map(lambda kv: kv))]:
    print(f"{name:35} {describe(r)}")

## 3. Execution: jobs, stages, tasks, shuffle

In [ ]:
moby = sc.textFile(BOOK, 4).flatMap(lambda line: line.lower().split())
cases = {
    "narrow only: map, filter": lambda: moby.map(len).filter(lambda n: n > 10).count(),
    "reduceByKey": lambda: pairs_counts.count(),
    "the same count again": lambda: pairs_counts.count(),
    "reduceByKey, then sortBy": lambda: moby.map(lambda w: (w, 1)).reduceByKey(add).sortBy(lambda kv: -kv[1]).take(3),
}
pairs_counts = moby.map(lambda w: (w, 1)).reduceByKey(add)
for name, run in cases.items():
    with Measure() as m:
        run()
    print(f"{name:28} {m}")

`reduceByKey` combines on the map side, `groupByKey` ships every value (shuffle compressed with lz4, the default; `02_stages_shuffle.py` also measures it uncompressed):

In [ ]:
pairs = sc.union([sc.textFile(BOOK, 4)] * 20).coalesce(4).flatMap(lambda l: l.lower().split()).map(lambda w: (w, 1))
with Measure() as r:
    pairs.reduceByKey(add).count()
with Measure() as g:
    pairs.groupByKey().mapValues(sum).count()
print("reduceByKey:", mb(r.shuffle_write), "| groupByKey:", mb(g.shuffle_write))

A join of co-partitioned RDDs needs no shuffle:

In [ ]:
a = moby.map(lambda w: (w, 1)).reduceByKey(add, 8)
b = moby.map(lambda w: (w, len(w))).distinct(6)                 # another partitioning
with Measure() as m:
    a.join(b).count()
print("not co-partitioned:", m)

a = moby.map(lambda w: (w, 1)).reduceByKey(add, 8).cache()
b = moby.map(lambda w: (w, len(w))).reduceByKey(max, 8).cache()
a.count(), b.count()
with Measure() as m:
    a.join(b).count()
print("co-partitioned:    ", m)

## 4. Fault tolerance: `checkpoint()` cuts the lineage

In [ ]:
sc.setCheckpointDir("data/checkpoints")
c = moby.map(lambda w: (w, 1)).reduceByKey(add).cache()
c.checkpoint()                                    # before the first action
c.count()
print(c.toDebugString().decode())

## 5. `cache` and `persist`

In [ ]:
def parse(line):
    page, links = line.split("\t")
    return int(page), [int(x) for x in links.split(",")]

calls = sc.accumulator(0)                         # counts the calls to parse
def parse_counted(line):
    calls.add(1)
    return parse(line)

links = sc.textFile(GRAPH, 8).map(parse_counted)  # no cache
for i in (1, 2):
    with Measure() as m:
        links.count()
    print(f"count {i}: parse called {calls.value:,} times so far, {m.seconds:.2f} s")

In [ ]:
calls = sc.accumulator(0)
links = sc.textFile(GRAPH, 8).map(parse_counted).cache()
print("before any action:", storage())
for i in (1, 2, 3):
    with Measure() as m:
        links.count()
    print(f"count {i}: parse called {calls.value:,} times so far, {m.seconds:.2f} s")
print(storage())
links.unpersist()

In [ ]:
from pyspark import StorageLevel

for name in ("MEMORY_ONLY", "MEMORY_AND_DISK", "DISK_ONLY", "MEMORY_ONLY_2"):
    print(f"{name:16}", getattr(StorageLevel, name))

A DataFrame cache: Catalyst reads it (`InMemoryTableScan`). The 512 MB example of the slides needs a fresh JVM: run `03b_cache_too_big.py`.

In [ ]:
df = spark.read.text(BOOK)
df.cache()
print(df.storageLevel)
df.count()
df.filter("value like '%whale%'").explain()

## 6. Closures and shared variables

In [ ]:
counter = 0
def increment(x):
    global counter
    counter += x
sc.parallelize(range(1, 16), 4).foreach(increment)
print("counter =", counter)

acc = sc.accumulator(0)
sc.parallelize(range(1, 16), 4).foreach(lambda x: acc.add(x))
print("acc.value =", acc.value)

In [ ]:
clients = sc.textFile("data/clients.csv").map(lambda l: l.split(",")).map(lambda f: (f[0], (f[1], f[2])))
commandes = sc.textFile("data/commandes.csv").map(lambda l: l.split(",")).map(lambda f: (f[1], (f[0], f[2], float(f[3]))))
with Measure() as m:
    print(sorted(commandes.join(clients).collect()))
print("reduce-side join:", m)

small_table = sc.broadcast(clients.collectAsMap())
with Measure() as m:
    print(sorted(commandes.map(lambda kv: (kv[0], (kv[1], small_table.value[kv[0]]))).collect()))
print("broadcast join:  ", m)

## 7. PageRank

In [ ]:
def contributions(pair):
    page, (links, rank) = pair
    yield page, 0.0                               # every page keeps a rank
    for dest in links:
        yield dest, rank / len(links)

def iteration(links, ranks, partitions):
    return (links.join(ranks).flatMap(contributions)
                 .reduceByKey(add, partitions)
                 .mapValues(lambda s: 0.15 + 0.85 * s))

toy = sc.parallelize([(1, [2]), (2, [1, 3]), (3, [1, 4]), (4, [1])]).partitionBy(2).cache()
ranks = toy.mapValues(lambda _: 1.0)
for i in range(1, 18):
    ranks = iteration(toy, ranks, 2).cache()
    if i in (1, 2, 17):
        print(f"iteration {i:2}:", "  ".join(f"{p}: {r:.3f}" for p, r in sorted(ranks.collect())))

The 500,000 pages, three versions (the slides used 10 iterations):

In [ ]:
ITERATIONS, P = 3, 8
for version in ("links not cached", "links cached", "links partitioned and cached"):
    g = sc.textFile(GRAPH, P).map(parse)
    if version == "links cached":
        g = g.cache()
    elif version == "links partitioned and cached":
        g = g.partitionBy(P).cache()
    g.count()
    ranks, times, shuffle = g.mapValues(lambda _: 1.0), [], []
    for i in range(ITERATIONS):
        new = iteration(g, ranks, P).cache()
        with Measure() as m:
            new.count()
        ranks.unpersist(); ranks = new
        times.append(m.seconds); shuffle.append(m.shuffle_write)
    print(f"{version:30} {sum(times) / ITERATIONS:5.2f} s and {mb(sum(shuffle) / ITERATIONS)} shuffled per iteration")
    g.unpersist(); ranks.unpersist()

GraphFrames: PageRank in one line.

In [ ]:
import glob
sc.addPyFile(glob.glob(os.path.expanduser("~/.ivy2/jars/graphframes_graphframes-*.jar"))[0])
from graphframes import GraphFrame

v = spark.createDataFrame([(1,), (2,), (3,), (4,)], ["id"])
e = spark.createDataFrame([(1, 2), (2, 1), (2, 3), (3, 1), (3, 4), (4, 1)], ["src", "dst"])
GraphFrame(v, e).pageRank(resetProbability=0.15, maxIter=17).vertices.orderBy("id").show()

## 8. DataFrames, SQL and Catalyst

In [ ]:
from pyspark.sql.functions import col, desc, explode, expr, lower, split, udf
from pyspark.sql.types import StringType

lines_df = spark.read.text(BOOK)
words_df = lines_df.select(explode(split(lower(col("value")), " ")).alias("word"))
counts_df = words_df.filter(words_df.word != "").groupBy("word").count().orderBy(desc("count"))
counts_df.show(5)

lines_df.createOrReplaceTempView("lines")
sql_df = spark.sql("""SELECT word, COUNT(*) AS count
                      FROM (SELECT explode(split(lower(value), ' ')) AS word FROM lines)
                      WHERE word != '' GROUP BY word ORDER BY count DESC""")
counts_df.explain()
sql_df.explain()                                   # the same physical plan

Movie query 3 on Parquet: the four plans, the filter pushed into the reader, two columns read.

In [ ]:
spark.read.option("multiLine", True).json("data/movies").withColumn("year", col("year").cast("int")) \
     .write.mode("overwrite").parquet("data/movies.parquet")
movies = spark.read.parquet("data/movies.parquet")
q3 = movies.filter(col("year") > 2000).select("title", "year")
q3.explain(True)
q3.show()

The optimizer moves the filter below the join and folds the constant; the join is a broadcast, or a sort-merge without broadcast:

In [ ]:
cl = spark.read.csv("data/clients.csv", schema="id INT, name STRING, city STRING")
co = spark.read.csv("data/commandes.csv", schema="id INT, id_client INT, product STRING, price DOUBLE")
def paris_orders():
    return (co.join(cl, co.id_client == cl.id).filter(col("city") == "Paris")
              .select("name", "product", expr("price * (1 + 20 / 100)").alias("price_ttc")))
paris_orders().explain(True)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
paris_orders().explain()                           # SortMergeJoin, two Exchanges
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10 * 1024 * 1024)

A Python UDF is a black box: no pushed filter, `BatchEvalPython`.

In [ ]:
upper_udf = udf(lambda s: s.upper(), StringType())
cl.filter(upper_udf(col("city")) == "PARIS").explain()
cl.filter(col("city") == "Paris").explain()

Adaptive query execution: the plan before and after running.

In [ ]:
agg = spark.read.text([BOOK] * 20).select(explode(split(lower(col("value")), " ")).alias("w")) \
           .filter(col("w") != "").groupBy("w").count()
agg.explain()
agg.collect()
agg.explain()                                      # isFinalPlan=true, AQEShuffleRead coalesced

RDD or DataFrame: the same word count on Moby Dick × 20.

In [ ]:
lines_rdd = sc.union([sc.textFile(BOOK, 4)] * 20)
big = spark.read.text([BOOK] * 20)
lower_udf = udf(lambda s: s.lower(), StringType())
runs = {
    "RDD, Python lambdas": lambda: lines_rdd.flatMap(lambda l: l.lower().split(" ")).filter(lambda w: w != "")
                                            .map(lambda w: (w, 1)).reduceByKey(add).count(),
    "DataFrame, built-ins": lambda: big.select(explode(split(lower(col("value")), " ")).alias("w"))
                                       .filter(col("w") != "").groupBy("w").count().count(),
    "DataFrame, Python UDF": lambda: big.select(explode(split(lower_udf(col("value")), " ")).alias("w"))
                                        .filter(col("w") != "").groupBy("w").count().count(),
}
for name, run in runs.items():
    run()                                          # warm up
    t = time.time(); run()
    print(f"{name:24} {time.time() - t:.2f} s")

## 9. Exercises

1. Run `05_pagerank.py` on Managed Service for Apache Spark (ex-Dataproc): compare with the MapReduce table.
2. Add `ranks.checkpoint()` every 5 iterations to PageRank; compare `toDebugString()` before and after.
3. Write the 14 movie queries with DataFrames; with `explain()`, which filters reach the reader? The year is a string in the JSON files: what does a cast do to the pushdown?
4. Write one PageRank iteration with DataFrames (a join and a groupBy): where are the shuffles in the plan?

In [ ]:
spark.stop()